# Zadanie 5: programowanie genetyczne i regresja symboliczna

Termin realizacji: 18 maja 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zmodyfikuj przykład `pysr_demo.ipynb` tak, aby uczył się funkcji $f(x) = 2.2\sin(x_0 + 2 x_1) - x_5^2 - 3$ której dziedziną jest $\mathbb{R}^6$. Uczenie ma się odbywać w oparciu o 200 wylosowanych z dziedziny próbek (między -5 a 5).
2. Zanotuj wzory trzech rozwiązań o najwyższej wartości `score` oraz rozwiązanie `best` dla następujących zestawów ustawień:
   1. `binary_operators=["+", "*"], unary_operators=["cos", "exp", "sin"], maxsize=20`,
   2. `binary_operators=["+", "*", "-", "^"], unary_operators=["cos", "exp", "sin", "log"], maxsize=30`, (dodaj ograniczenie dla argumentów operatora "^": [https://astroautomata.com/PySR/v1.5.9/options.html#constraining-use-of-operators](https://astroautomata.com/PySR/v1.5.9/options.html#constraining-use-of-operators).
   3. `binary_operators=["+", "*", "-", "^"], unary_operators=["exp", "sin"], maxsize=15`.
3. Powtórz eksperymenty z zadania na 3.0 po dodaniu szumu do próbek z funkcji $f$ (rozkład normalny o średniej 0 i odchyleniu standardowym 0.5)

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Dodaj do porównania dopasowanie oparte o próbki losowane w szerszym zakresie (między -15 a 15) oraz wyższy poziom szumu (odchylenie standardowe równe 2 oraz 5).

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zamień funkcję $f$ na $f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - p(\lfloor x_0 \rfloor)$ gdzie $p(i)$ oznacza $i$-tą liczbę pierwszą. Uwzględnij `p` jako dodatkowy operator unarny analogicznie do przykładu "Julia packages and types" z notatnika `pysr_demo.ipynb`. Powtórz eksperymenty opisane w zadaniach na 3.0 i 4.0.


# Zadanie 1 (Na 3.0)

## Ekspeymenty bez sumu

In [2]:
import pysr
import sympy
import numpy as np
from matplotlib import pyplot as plt
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split

np.random.seed(0)
X = np.random.uniform(-5, 5, size=(200, 6))
y = 2.2 * np.sin(X[:, 0]+ 2*X[:,1]) - X[:,5]**2 - 3

In [3]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    verbosity=False,
    **default_pysr_params,
)
model.fit(X, y)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

/home/tk2/projekty/MetodyOptymalizacji/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 11 (Score: 13.7280):
Wzór: x5*x5*(-1.0) + sin(x0 + x1 + x1)*2.2 - 3.0
------------------------------
Miejsce 4 (Score: 2.0312):
Wzór: x5*(-1.1931677)*x5
------------------------------
Miejsce 10 (Score: 1.1394):
Wzór: x5*(-0.9899335)*x5 + sin(x0 + x1 + x1) - 3.0889792
------------------------------


x5*x5*(-1.0) + sin(x0 + x1 + x1)*2.2 - 3.0

In [9]:
model = PySRRegressor(
    niterations=100,
    binary_operators=[
        "+", "*", "-",
        "my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"
    ],
    extra_sympy_mappings={
        "my_pow": lambda x, y: x**y
    },
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    verbosity=False,
    constraints={'my_pow': (-1,1)},
    **default_pysr_params,
)
model.fit(X, y)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

/home/tk2/projekty/MetodyOptymalizacji/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 11 (Score: 27.2189):
Wzór: -x5*x5 + sin(x0 + x1 + x1)*2.2 - 3.0
------------------------------
Miejsce 4 (Score: 3.1427):
Wzór: -x5*x5 - 3.0129473
------------------------------
Miejsce 9 (Score: 1.1454):
Wzór: -x5*x5 + sin(x0 + x1 + x1) - 3.0070155
------------------------------


-x5*x5 + sin(x0 + x1 + x1)*2.2 - 3.0

In [11]:
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*", "-","my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"],
    extra_sympy_mappings={
        "my_pow": lambda x, y: x**y
    },
    unary_operators=["exp", "sin"],
    maxsize=15,
    verbosity=False,
    constraints={'my_pow': (-1,1)},
    **default_pysr_params,
)
model.fit(X, y)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

/home/tk2/projekty/MetodyOptymalizacji/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 11 (Score: 27.2399):
Wzór: -x5*x5 + sin(x0 + x1 + x1)*2.2 - 3.0000002
------------------------------
Miejsce 4 (Score: 3.2011):
Wzór: -x5*x5 - 3.0129473
------------------------------
Miejsce 9 (Score: 1.1391):
Wzór: -x5*x5 + sin(x0 + x1*2.002528) - 3.006651
------------------------------


-x5*x5 + sin(x0 + x1 + x1)*2.2 - 3.0000002

## Eksperymenty z szumem

In [13]:
np.random.seed(0)
X = np.random.uniform(-5, 5, size=(200, 6))
y = 2.2 * np.sin(X[:, 0]+ 2*X[:,1]) - X[:,5]**2 - 3
noise = 0.5 * np.random.randn(200)
y_noised = y + noise

In [14]:
default_pysr_params = dict(
    populations=30,
    model_selection="best",
)
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    verbosity=False,
    **default_pysr_params,
)
model.fit(X, y_noised)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

/home/tk2/projekty/MetodyOptymalizacji/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 4 (Score: 2.0183):
Wzór: x5*x5*(-1.1879632)
------------------------------
Miejsce 12 (Score: 1.3250):
Wzór: x5*(-0.99840987)*x5 + sin(x0 + x1 + x1)*2.215479 - 2.9473767
------------------------------
Miejsce 10 (Score: 0.9533):
Wzór: x5*(-0.98821336)*x5 + sin(x0 + x1 + x1) - 3.0375047
------------------------------


x5*(-0.99840987)*x5 + sin(x0 + x1 + x1)*2.215479 - 2.9473767

In [15]:
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*", "-","my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"],
    extra_sympy_mappings={
        "my_pow": lambda x, y: x**y
    },
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    verbosity=False,
    constraints={'pow': (-1,1)},
    **default_pysr_params,
)
model.fit(X, y_noised)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

/home/tk2/projekty/MetodyOptymalizacji/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 4 (Score: 3.0386):
Wzór: -x5*x5 - 2.9474645
------------------------------
Miejsce 9 (Score: 0.9575):
Wzór: -(x5*x5 + sin(-x0 + x1*(-1.9819049))) - 2.9439185
------------------------------
Miejsce 10 (Score: 0.7377):
Wzór: -(x5*x5 + sin(-x0 + x1*(-1.9806921))*2.2229629) - 2.9401243
------------------------------


-(x5*x5 + sin(-x0 + x1*(-1.9806921))*2.2229629) - 2.9401243

In [16]:
model = PySRRegressor(
    niterations=100,
    binary_operators=["+", "*", "-","my_pow(x,y) = (x > 0) ? x^y : convert(typeof(x), NaN)"],
    extra_sympy_mappings={
        "my_pow": lambda x, y: x**y
    },
    unary_operators=["exp", "sin"],
    maxsize=15,
    verbosity=False,
    constraints={'pow': (-1,1)},
    **default_pysr_params,
)
model.fit(X, y_noised)
eqs = model.equations_
top_3_scores = eqs.sort_values("score",ascending=False).head(3)
for i, row in top_3_scores.iterrows():
    print(f"Miejsce {i+1} (Score: {row['score']:.4f}):")
    print(f"Wzór: {row['sympy_format']}")
    print("-"*30)
model.sympy()

/home/tk2/projekty/MetodyOptymalizacji/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


Miejsce 4 (Score: 3.0967):
Wzór: -x5*x5 - 2.9474645
------------------------------
Miejsce 9 (Score: 0.9574):
Wzór: -x5*x5 + sin(x0 + x1*1.9818705) - 1*2.9438612
------------------------------
Miejsce 8 (Score: 0.0180):
Wzór: -x5*x5 + sin(sin(x1*(-2.0216112))) - 1*3.0098364
------------------------------


-x5*x5 + sin(x0 + x1*1.9818705) - 1*2.9438612

# Zadanie na 4.0